In [1]:
# std
import concurrent
import concurrent.futures
import copy
import datetime
import hashlib
import inspect
import json
import logging
import multiprocessing
import os
import psutil
import sys
import shutil
import time
import warnings
from pathlib import Path

import IPython
import IPython.display as ipyd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as scp
import scipy.stats as stats

warnings.filterwarnings('ignore')
logging.basicConfig(format='[%(levelname)-4s]  %(message)s', level=logging.INFO)
sns.set_style('darkgrid')

%config InlineBackend.figure_format = 'retina'
%config InteractiveShell.ast_node_interactivity = 'all'
%config TerminalInteractiveShell.cache_size = 0

pd.options.display.max_columns = int(1e5)
max_workers = int(multiprocessing.cpu_count())

sys.version
multiprocessing.cpu_count(), psutil.cpu_freq()


(16, scpufreq(current=4056, min=1020, max=4056))

In [2]:
from pprint import pprint

def dist_from_sample(sample):
  mean = np.array(sample).mean()
  std = np.array(sample).std(ddof=1)
  return (mean, std)


In [9]:
import math

files = list(Path('perfs').glob('*.json'))
all_items = {}

selectors = {
  'ninja_build_time',
}


bold_selectors = {
  'local_build_perf_metrics/ninja_build_time/full_build_all_target_build_all',
  'public_changes',
}

def build_graphs(name='comparison', filter={}):
  for perf in ['results_our', 'results_scipy']:
      data = json.loads(Path(perf).read_text())
      for k, v in data.items():
        tmp = all_items.get(k, {})
        tmp[perf] = v
        all_items[k] = tmp

  n = len(all_items)
  cols = 4
  rows = math.ceil(n / cols)
  plt.figure(figsize=(4 * 4 * 2.5, 4 * 1.7 * rows))

  for i, (k, v) in enumerate(all_items.items()):
    _ = plt.subplot(rows, cols, i + 1)
    samples = {}
    for (perf_name, sample) in sorted(v.items()):
      _ = sns.distplot(sample, label=perf_name, kde=True)
    _ = plt.title(k)
    _ = plt.legend()

  plt.tight_layout()
  plt.savefig(f'{name}.pdf')

In [11]:
build_graphs()